# Combining error detection with error cancellation

Spacetime Pauli checks detect errors and post-selection removes them, but only the errors the checks can see. Probabilistic error cancellation (PEC) removes the rest, at a sampling cost that is exponential in the total noise it has to invert. Combined, the two divide the noise between them: `qiskit-mitigation`'s `create_postselected_noise_mask` zeroes every noise generator the checks detect, so PEC inverts only the undetected remainder and its sampling overhead $\gamma^2$ drops to $\gamma'^2$, typically by orders of magnitude for a well-checked circuit. This guide shows how a `CheckedCircuit` plugs into that machinery; nothing beyond `qiskit-paulice`'s checks, `samplomatic`, and `qiskit-mitigation` is needed.

The combination comes with two concessions, both of which this guide makes explicit:

1. **One noise layer per check.** A check's entangling gate gets a box of its own, so the noise learner characterizes the payload's layers plus one small two-qubit layer per check. `CheckedCircuit.box` lays the circuit out this way.
2. **The truncation is first order.** A generator is zeroed because post-selection removes it *on its own*; two detected generators that fire together can multiply into something the checks miss, which PEC then does not cancel either. The residual bias is $O(\lambda^2)$ in the noise rates and grows faster than linearly with circuit depth. The last section measures it.

To keep the focus on the estimator rather than on a device, the circuit is simulated with Qiskit Aer and the noise model used for mitigation *is* the noise applied, so every deviation from the exact answer is the method's own. On hardware the maps come from `NoiseLearnerV3` instead; everything else is unchanged.

**Workflow**
1. Build a Clifford circuit whose exact expectation values are known.
2. Add checks and box the circuit for noise learning and sampling.
3. Shade the noise model by detectability and compare the sampling overheads.
4. Sample the twirled circuit with and without noise inversion.
5. Post-select, apply the PEC signs, and estimate.
6. Measure the first-order truncation bias.

## 1. A circuit with a known answer

A brickwork Clifford circuit followed by its inverse returns every qubit to $|0\rangle$, so $\langle Z_q \rangle = 1$ exactly for every qubit while the noisy circuit is as deep and as entangling as any other. The two unique entangling layers (even and odd edges) are recorded as `payload_layers` so that boxing can reproduce them.

In [1]:
import warnings

import numpy as np
from qiskit import QuantumCircuit

NUM_QUBITS, FORWARD_LAYERS, SEED = 10, 6, 1234
rng = np.random.default_rng(SEED)


def mirror_brickwork(num_qubits, num_layers, rng):
    """A random brickwork Clifford circuit followed by its inverse, then measured."""
    forward = QuantumCircuit(num_qubits)
    for layer in range(num_layers):
        for qubit in range(num_qubits):
            if rng.integers(0, 2):
                forward.sx(qubit)
            if rng.integers(0, 2):
                forward.s(qubit)
        for qubit in range(layer % 2, num_qubits - 1, 2):
            forward.cz(qubit, qubit + 1)
    circuit = forward.compose(forward.inverse())
    circuit.measure_all()
    return circuit


circuit = mirror_brickwork(NUM_QUBITS, FORWARD_LAYERS, rng)
payload_layers = [
    {(i, i + 1) for i in range(0, NUM_QUBITS - 1, 2)},
    {(i, i + 1) for i in range(1, NUM_QUBITS - 1, 2)},
]
circuit.draw("mpl", fold=-1)

<Figure size 3590.39x953.167 with 1 Axes>

## 2. Add checks and box the circuit

`add_pauli_checks` returns one `CheckedCircuit` per number of checks; we take them all. `CheckedCircuit.box` then groups the executed circuit (ancillas and check gates included) into the annotated boxes `samplomatic` uses for twirling, noise learning and sampling. Each check gate is scheduled into a box of its own, so the number of unique layers to learn is the payload's two plus one per check, however deep the circuit.

In [2]:
from qiskit_paulice import add_pauli_checks
from qiskit_paulice.noise_models import NoiseModel
from samplomatic.annotations import InjectNoise
from samplomatic.utils import get_annotation

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    checked = add_pauli_checks(
        circuit,
        list(range(NUM_QUBITS)),
        NoiseModel(gate_noise=3e-3, readout_noise=1e-2),
        seed=SEED,
    )[-1]

boxed = checked.box(payload_layers=payload_layers)
refs = [
    inject.ref
    for instruction in boxed.data
    if instruction.operation.name == "box"
    and (inject := get_annotation(instruction.operation, InjectNoise)) is not None
    and inject.ref
]
print(
    f"{len(checked.check_support)} checks; {len(refs)} noise boxes, {len(set(refs))} unique layers to learn"
)

10 checks; 43 noise boxes, 12 unique layers to learn


## 3. Shade the noise model by detectability

A learner returns one `PauliLindbladMap` per unique layer, keyed by the box's `InjectNoise.ref` and indexed in the box's physical qubit order. Here a synthetic model stands in for it: single-qubit generators on every qubit of a layer and all fifteen two-qubit generators on every entangled pair, with two-qubit process infidelities around $3\times10^{-3}$.

`create_postselected_noise_mask` takes the boxed circuit, the maps, and a list of *detectors*: Pauli operators at the end of the circuit whose outcomes post-selection fixes. It propagates each detector backward through the circuit and masks every generator that anticommutes with one at its injection point. A spacetime check's syndrome is the parity of the measured bits in its `check_support`, so its detector is the $Z$ string on that support. The mask is what `samplomatic` calls `local_scales`; passing it alongside `noise_scales = -1` makes the samplex invert only the undetected generators. The function also returns the sampling cost $\gamma'^2$ of that inversion, where $\gamma = \exp(2\sum_j \lambda_j)$ is summed over box *instances*, so a repeated layer is paid for every time it occurs.

Two things to get right. The circuit to shade is the boxed *checked* circuit, the one that is executed, not a boxing of the bare circuit: check insertion changes which gates share a layer, so the bare circuit's boxes carry different `ref` values and colliding `modifier_ref` values. And the mask is a first-order statement, exact for one generator at a time; the last section measures what that costs.

In [3]:
from qiskit.quantum_info import PauliLindbladMap, PauliList
from qiskit_mitigation.noise import create_postselected_noise_mask


def check_detectors(checked):
    """The Z string on each check's support: the operator whose outcome post-selection fixes."""
    num_qubits = checked.circuit.num_qubits
    return PauliList(
        [
            "".join("Z" if q in support else "I" for q in range(num_qubits))[::-1]
            for support in checked.check_support
        ]
    )


def synthetic_noise(boxed, rng, one_qubit_rate=5e-5, two_qubit_rate=2e-4):
    """A stand-in for a learned model: one Pauli-Lindblad map per unique layer of ``boxed``."""
    rates = {}
    for instruction in boxed.data:
        if instruction.operation.name != "box":
            continue
        inject = get_annotation(instruction.operation, InjectNoise)
        if inject is None or not inject.ref or inject.ref in rates:
            continue
        body = instruction.operation.blocks[0]
        physical = [boxed.find_bit(q).index for q in instruction.qubits]
        # a map indexes the box's qubits in physical order
        local = {i: rank for rank, i in enumerate(np.argsort(physical))}
        terms = []
        for i in range(len(physical)):
            terms += [(p, (local[i],), one_qubit_rate * rng.uniform(0.5, 1.5)) for p in "XYZ"]
        for sub in body.data:
            if len(sub.qubits) == 2:
                a, b = (local[body.find_bit(q).index] for q in sub.qubits)
                terms += [
                    (p + r, (a, b), two_qubit_rate * rng.uniform(0.5, 1.5))
                    for p in "IXYZ"
                    for r in "IXYZ"
                    if p + r != "II"
                ]
        rates[inject.ref] = PauliLindbladMap.from_sparse_list(terms, num_qubits=len(physical))
    return rates


def gamma(boxed, rates, local_scales=None):
    """The PEC sampling overhead of inverting ``rates`` over every noise box of ``boxed``."""
    total = 0.0
    for instruction in boxed.data:
        if instruction.operation.name != "box":
            continue
        inject = get_annotation(instruction.operation, InjectNoise)
        if inject is None or not inject.ref:
            continue
        generator_rates = np.asarray(rates[inject.ref].rates)
        if local_scales is not None:
            generator_rates = generator_rates * local_scales[inject.modifier_ref]
        total += generator_rates.sum()
    return float(np.exp(2 * total))


rates = synthetic_noise(boxed, rng)
local_scales, cost = create_postselected_noise_mask(boxed, rates, check_detectors(checked))
detected = 1 - np.concatenate(list(local_scales.values())).mean()
gamma_full, gamma_reduced = gamma(boxed, rates), np.sqrt(cost)
print(f"The checks detect {detected:.1%} of the generators.")
print(f"PEC alone pays gamma = {gamma_full:.2f}; after the checks, gamma' = {gamma_reduced:.2f}.")

The checks detect 84.7% of the generators.
PEC alone pays gamma = 1.74; after the checks, gamma' = 1.08.


## 4. Sample the twirled circuit

`samplomatic.build` turns the boxed circuit into a parameterized template and a samplex that draws twirls, basis changes and, when `noise_scales` is nonzero, the Pauli insertions and signs that invert the noise. The noise itself is applied by appending each layer's `PauliLindbladError` after the barrier that closes its box in every bound circuit, and the Clifford circuits are run on Aer's stabilizer simulator.

Three sample sets cover four methods: the baseline with no inversion serves both raw and QED, PEC inverts every generator, and QED + PEC inverts only the undetected ones. A PEC estimate's precision is set by the number of randomizations rather than shots, since every shot in a randomization shares one sign, so the inverting arms get many randomizations of few shots.

In [4]:
import samplomatic
from qiskit.quantum_info import Pauli
from qiskit_aer import AerSimulator
from qiskit_aer.noise import PauliLindbladError

template, samplex = samplomatic.build(boxed)


def layer_errors(boxed, rates):
    """One Aer ``PauliLindbladError`` per noise box of ``boxed``, keyed by box ordinal."""
    errors, box_index = {}, -1
    for instruction in boxed.data:
        if instruction.operation.name != "box":
            continue
        box_index += 1
        inject = get_annotation(instruction.operation, InjectNoise)
        if inject is None or not inject.ref:
            continue
        qubits = sorted(boxed.find_bit(q).index for q in instruction.qubits)
        paulis, lambdas = [], []
        for label, indices, rate in rates[inject.ref].to_sparse_list():
            chars = ["I"] * len(qubits)
            for char, i in zip(label, indices, strict=True):
                chars[len(qubits) - 1 - i] = char
            paulis.append(Pauli("".join(chars)))
            lambdas.append(float(rate))
        errors[box_index] = (PauliLindbladError(paulis, lambdas).to_instruction(), qubits)
    return errors


def apply_noise(bound, errors):
    """Append each box's error after the barrier that closes it in a bound template."""
    out, box_index = bound.copy_empty_like(), -1
    for instruction in bound.data:
        out.append(instruction)
        label = instruction.operation.label or ""
        if instruction.operation.name == "barrier" and label.startswith("R"):
            box_index += 1
            if box_index in errors:
                out.append(*errors[box_index])
    return out


def to_clifford(bound):
    """Rewrite the twirl's ``rz(k pi/2)`` gates as Clifford gates for the stabilizer simulator."""
    out = bound.copy_empty_like()
    for instruction in bound.data:
        if instruction.operation.name == "rz":
            k = round(float(instruction.operation.params[0]) / (np.pi / 2)) % 4
            if k:
                getattr(out, ["id", "s", "z", "sdg"][k])(instruction.qubits[0])
        else:
            out.append(instruction)
    return out


errors = layer_errors(boxed, rates)
simulator = AerSimulator(method="stabilizer", seed_simulator=SEED)
z_basis = {
    spec.name.split(".", 1)[1]: np.ones(spec.shape, dtype=np.uint8)
    for spec in samplex.inputs().get_specs("basis_changes")
}


def sample(noise_scale, local_scales, num_randomizations, shots, seed):
    """Draw randomizations from the samplex and run them on the noisy simulator."""
    inputs = samplex.inputs().bind(
        pauli_lindblad_maps=rates,
        basis_changes=z_basis,
        local_scales=local_scales,
        **{spec.name: noise_scale for spec in samplex.inputs().get_specs("noise_scales")},
    )
    outputs = samplex.sample(inputs, num_randomizations=num_randomizations, rng=seed)
    circuits = [
        to_clifford(apply_noise(template.assign_parameters(values), errors))
        for values in outputs["parameter_values"]
    ]
    result = simulator.run(circuits, shots=shots, memory=True).result()
    bits = np.zeros((num_randomizations, shots, template.num_clbits), dtype=np.uint8)
    for r in range(num_randomizations):
        for s, shot in enumerate(result.get_memory(r)):
            bits[r, s] = [int(c) for c in shot.replace(" ", "")[::-1]]
    return outputs, bits


ones = {modifier: np.ones_like(mask) for modifier, mask in local_scales.items()}
runs = {
    "baseline": sample(0.0, ones, num_randomizations=256, shots=64, seed=1),
    "pec": sample(-1.0, ones, num_randomizations=4096, shots=8, seed=2),
    "qed_pec": sample(-1.0, local_scales, num_randomizations=4096, shots=8, seed=3),
}

## 5. Post-select, sign, and estimate

Twirling flips measured bits deterministically within each randomization, and the samplex reports which (`measurement_flips.<register>`). Every measured bit, payload and ancilla alike, is XORed with its flip before anything else is done with it. Post-selecting on the raw bits instead would keep exactly the wrong shots in every randomization whose twirl flips a check, and nothing about the result would look wrong. With the flips undone, `CheckedCircuit.get_postselection_method` gives the syndrome of each shot as usual (`add_pauli_checks` measures qubit $q$ into classical bit $q$, so the arrays can be passed to it directly).

The expectation values come from `qiskit-mitigation`'s `executor_expectation_values`, which applies the PEC signs and the post-selection mask together. The two do not simply compose: the kept fraction is itself a quantity PEC has to estimate, so the estimator is a ratio of sign-weighted sums whose common overhead cancels. Its error bars are taken from a bootstrap over randomizations rather than from the per-shot variance: the shots of one randomization share its sign and its inserted Paulis, so they are not independent samples.

In [5]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_mitigation.utils import executor_expectation_values

payload = [checked.circuit.find_bit(clbit).index for clbit in checked.circuit.cregs[0]]
postselect = checked.get_postselection_method()
per_qubit = [SparsePauliOp("I" * (NUM_QUBITS - 1 - q) + "Z" + "I" * q) for q in range(NUM_QUBITS)]
basis_map = {Pauli("Z" * NUM_QUBITS): [*per_qubit, sum(per_qubit) / NUM_QUBITS]}


def undo_twirl(outputs, bits):
    """XOR every measured bit with the flip the twirl put on it."""
    flips = np.zeros_like(bits)
    for creg in checked.circuit.cregs:
        block = np.asarray(outputs[f"measurement_flips.{creg.name}"], dtype=np.uint8)
        for i, clbit in enumerate(creg):
            flips[..., checked.circuit.find_bit(clbit).index] = block[..., i]
    return bits ^ flips


def estimate(outputs, bits, postselect_checks, pec, bootstraps=200, seed=0):
    """Per-qubit <Z> and their pooled mean, with bootstrap standard errors, for one sample set."""
    bits = undo_twirl(outputs, bits)
    accepted = np.ones(bits.shape[:2], dtype=bool)
    if postselect_checks:
        accepted = ~np.array([[postselect(shot).any() for shot in rand] for rand in bits])
    bits = bits[..., payload].astype(bool)
    signs = np.asarray(outputs["pauli_signs"]).reshape(len(bits), -1) if pec else None

    def point(index):
        results = executor_expectation_values(
            bits[index],
            basis_map,
            None,
            avg_axis=0,
            pauli_signs=None if signs is None else signs[index],
            postselect_mask=accepted[index],
        )
        return np.array([value for value, _ in results])

    values = point(np.arange(len(bits)))
    # The shots of one randomization share its sign and its inserted Paulis, so the independent
    # units are the randomizations: resample those.
    rng = np.random.default_rng(seed)
    replicas = [point(rng.integers(0, len(bits), len(bits))) for _ in range(bootstraps)]
    errors = np.std(replicas, axis=0)
    return values[:-1], errors[:-1], values[-1], errors[-1], accepted.mean()


results = {
    "raw": estimate(*runs["baseline"], postselect_checks=False, pec=False),
    "QED": estimate(*runs["baseline"], postselect_checks=True, pec=False),
    "PEC": estimate(*runs["pec"], postselect_checks=False, pec=True),
    "QED + PEC": estimate(*runs["qed_pec"], postselect_checks=True, pec=True),
}
for name, (_, _, value, error, accepted_fraction) in results.items():
    print(f"{name:10s} {value:.4f} ± {error:.4f}   accepted {accepted_fraction:.1%}")

raw        0.9123 ± 0.0014   accepted 100.0%
QED        0.9956 ± 0.0003   accepted 78.9%
PEC        0.9842 ± 0.0059   accepted 100.0%
QED + PEC  0.9999 ± 0.0012   accepted 79.1%


In [6]:
import matplotlib.pyplot as plt

styles = {"raw": ("o", "C0"), "QED": ("s", "C1"), "PEC": ("^", "C4"), "QED + PEC": ("D", "C2")}
x = np.arange(NUM_QUBITS)
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, (values, errors, value, error, _) in results.items():
    marker, color = styles[name]
    ax.errorbar(
        x,
        values,
        errors,
        fmt=marker,
        color=color,
        mfc="white",
        capsize=3,
        label=f"{name}: {value:.4f} ± {error:.4f}",
    )
ax.axhline(1.0, color="black", lw=1)
ax.set(
    xlabel="payload qubit",
    ylabel=r"$\langle Z \rangle$",
    xticks=x,
    title=f"{NUM_QUBITS} qubits, {2 * FORWARD_LAYERS} entangling layers, {len(checked.check_support)} checks",
)
ax.legend(loc="lower right");

Detection alone recovers most of the signal; inverting the undetected remainder closes the gap to the exact value, at a fraction of plain PEC's cost: the QED + PEC arm pays $\gamma'^2$ instead of $\gamma^2$ and its error bar shows it. Plain PEC, with the same number of randomizations, is still visibly noisy.

## 6. The first-order truncation bias

The mask zeroes a generator on the grounds that post-selection removes it. That is exact for one generator at a time, but a shot carries every generator that fired, and two detected generators can combine into an error the checks do not see. PEC then cancels neither, and the estimate is biased by $O(\lambda^2)$.

Because the circuit is Clifford, the effect of a set of errors on every check and every payload $Z$ is the XOR of the individual effects, so the bias can be computed exactly under the model without simulating any circuit: propagate the checks and the payload $Z$ operators back to each injection point once per generator, then draw error configurations and accumulate their flip patterns. Dividing the accepted shots' $\langle Z \rangle$ by what the first-order inversion cancels gives the QED + PEC estimate the mitigation converges to, and its deviation from one is the bias.

In [7]:
def generator_signatures(checked, boxed, rates, local_scales):
    """Each generator instance's flip pattern on every check and payload Z, as a bitmask.

    Bit ``i < num_checks`` is check ``i``; bit ``num_checks + q`` is payload qubit ``q``.
    """
    n, num_checks = boxed.num_qubits, len(checked.check_support)
    labels = [
        "".join("Z" if q in support else "I" for q in range(n))[::-1]
        for support in checked.check_support
    ]
    labels += ["".join("Z" if i == q else "I" for i in range(n))[::-1] for q in range(NUM_QUBITS)]
    detectors = PauliList(labels)
    signatures, lambdas, scales = [], [], []
    for instruction in reversed(boxed.data):
        qargs = [boxed.find_bit(q).index for q in instruction.qubits]
        if instruction.operation.name != "box":
            if instruction.operation.name not in ("measure", "barrier"):
                detectors = detectors.evolve(instruction.operation, qargs=qargs, frame="h")
            continue
        inject = get_annotation(instruction.operation, InjectNoise)
        if inject is not None and inject.ref:  # the noise is injected after the box, i.e. here
            qubits = sorted(qargs)
            for term, (label, indices, rate) in enumerate(rates[inject.ref].to_sparse_list()):
                chars = ["I"] * n
                for char, i in zip(label, indices, strict=True):
                    chars[n - 1 - qubits[i]] = char
                anticommutes = ~detectors.commutes(Pauli("".join(chars)))
                signatures.append(sum(1 << i for i, a in enumerate(anticommutes) if a))
                lambdas.append(rate)
                scales.append(local_scales[inject.modifier_ref][term])
        body = instruction.operation.blocks[0]
        for sub in reversed(body.data):
            if sub.operation.name not in ("measure", "barrier"):
                sub_qargs = [qargs[body.find_bit(q).index] for q in sub.qubits]
                detectors = detectors.evolve(sub.operation, qargs=sub_qargs, frame="h")
    return np.array(signatures, dtype=np.uint64), np.array(lambdas), np.array(scales), num_checks


def predicted_bias(signatures, lambdas, scales, num_checks, shots, rng):
    """Exact Pauli-level Monte Carlo of QED + first-order PEC under the model: the bias in percent."""
    accumulated = np.zeros(shots, dtype=np.uint64)
    for signature, probability in zip(signatures, (1 - np.exp(-2 * lambdas)) / 2, strict=True):
        count = rng.binomial(shots, probability)
        if count and signature:
            np.bitwise_xor.at(accumulated, rng.integers(0, shots, count), signature)
    kept = accumulated[(accumulated & np.uint64((1 << num_checks) - 1)) == 0]
    inverted = (1 - np.exp(-2 * lambdas * scales)) / 2  # what PEC cancels: the undetected share
    biases = []
    for q in range(NUM_QUBITS):
        bit = np.uint64(1 << (num_checks + q))
        value = 1 - 2 * ((kept & bit) != 0).mean()
        cancelled = np.prod(1 - 2 * inverted[(signatures & bit) != 0])
        biases.append(1 - value / cancelled)
    return 100 * np.mean(biases), len(kept) / shots


depths, biases = [], []
for forward_layers in (3, 6, 12, 24, 48):
    depth_rng = np.random.default_rng(SEED + forward_layers)
    circuit_l = mirror_brickwork(NUM_QUBITS, forward_layers, depth_rng)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        checked_l = add_pauli_checks(
            circuit_l,
            list(range(NUM_QUBITS)),
            NoiseModel(gate_noise=3e-3, readout_noise=1e-2),
            seed=SEED,
        )[-1]
    boxed_l = checked_l.box(payload_layers=payload_layers)
    rates_l = synthetic_noise(boxed_l, depth_rng)
    scales_l, _ = create_postselected_noise_mask(boxed_l, rates_l, check_detectors(checked_l))
    signatures = generator_signatures(checked_l, boxed_l, rates_l, scales_l)
    bias, acceptance = predicted_bias(*signatures, 4_000_000, depth_rng)
    depths.append(2 * forward_layers)
    biases.append(bias)
    print(f"L = {2 * forward_layers:3d}: accepted {acceptance:6.1%}, QED + PEC bias {bias:.3f} %")

depths, biases = np.array(depths), np.array(biases)
exponent, log_prefactor = np.polyfit(np.log(depths), np.log(biases), 1)
print(f"power-law fit: bias ∝ L^{exponent:.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
grid = np.linspace(depths.min(), depths.max() * 1.05, 100)
ax.plot(
    grid,
    np.exp(log_prefactor) * grid**exponent,
    "--",
    color="gray",
    label=f"fit: bias ∝ L$^{{{exponent:.2f}}}$",
)
ax.plot(depths, biases, "D", color="C2", label="predicted bias (exact Pauli-level MC)")
ax.set(
    xlabel="entangling layers L", ylabel="QED + PEC bias (%)", title="First-order truncation bias"
)
ax.legend();

L =   6: accepted  87.2%, QED + PEC bias 0.007 %


L =  12: accepted  78.8%, QED + PEC bias 0.014 %


L =  24: accepted  66.1%, QED + PEC bias 0.034 %


L =  48: accepted  44.0%, QED + PEC bias 0.107 %


L =  96: accepted  20.6%, QED + PEC bias 0.243 %
power-law fit: bias ∝ L^1.31


The bias is far below the statistical error of the run above, and it is second order in the rates: halving the noise quarters it. With depth it grows faster than linearly. The exponent depends on how often two detected generators can combine into an error the checks miss, which is a property of the checks' geometry; here it fits about $L^{1.3}$, and on a 20-qubit chain with a sparser set of checks it was found to be quadratic. Either way it is the quantity to budget against: once the statistical error of QED + PEC is pushed below it, deeper circuits need fewer undetected generators (more or better checks) or a less noisy device, not more shots. The same Monte Carlo doubles as a planning tool, since it costs seconds rather than a run.